#  Bias Evaluation on Meeting Summaries

This notebook computes bias metrics for each meeting summary.

In [2]:
import urllib3
print(urllib3.__version__)


2.0.7


In [1]:
import numpy as np

In [2]:
import json
import pandas as pd
import sys
import os
from textblob import TextBlob
sys.path.append(os.path.abspath("../src"))

from benchmark.eval_metrics import (
    polarity_shift, factual_consistency_score,
    lexical_diversity, rouge_score, bleu_score
)


with open('../data/public/meeting_examples.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)

###  Bias Metrics for Each Entry

In [ ]:
#a test 
import evaluate

bleu = evaluate.load("bleu")
pred = ["The cat is on the mat."]
ref = [["There is a cat on the mat."]] 
score = bleu.compute(predictions=pred, references=ref)
print(score)


Using the latest cached version of the module from C:\Users\EL mahjoubi\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--bleu\9e0985c1200e367cce45605ce0ecb5ede079894e0f24f54613fca08eeb8aff76 (last modified on Sun Jan 12 21:29:47 2025) since it couldn't be found locally at evaluate-metric--bleu, or remotely on the Hugging Face Hub.


{'bleu': 0.39442436483275556, 'precisions': [0.8571428571428571, 0.5, 0.4, 0.25], 'brevity_penalty': 0.8668778997501817, 'length_ratio': 0.875, 'translation_length': 7, 'reference_length': 8}


In [4]:
results = []
df['text'] = df['text'].apply(lambda x: " ".join(x) if isinstance(x, (list, np.ndarray)) else x)
df['summary'] = df['summary'].apply(lambda x: " ".join(x) if isinstance(x, (list, np.ndarray)) else x)


for _, row in df.iterrows():
    original = row['text']
    summary = row['summary']

    if isinstance(original, (list, np.ndarray)):
       original = " ".join(original)
    if isinstance(summary, (list, np.ndarray)):
       summary = " ".join(summary)

    metrics = {
        'text': original,
        'summary': summary,
        'polarity_shift': polarity_shift(original, summary),
        # 'factual_consistency': factual_consistency_score(original, summary),
        'lexical_diversity': lexical_diversity(summary),
        'rouge': rouge_score(original, summary)['rouge1'],
        'bleu': bleu_score(original, summary)['bleu']
    }
    results.append(metrics)

eval_df = pd.DataFrame(results)
eval_df


Using the latest cached version of the module from C:\Users\EL mahjoubi\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--rouge\b01e0accf3bd6dd24839b769a5fda24e14995071570870922c71970b3a6ed886 (last modified on Sun Jan 12 21:52:36 2025) since it couldn't be found locally at evaluate-metric--rouge, or remotely on the Hugging Face Hub.


,text,summary,polarity_shift,lexical_diversity,rouge,bleu
0,The product strategy meeting included a heated...,Sarah and Tom debated retention vs acquisition...,-0.25,1.0,0.315789,0.025377
1,"In the customer feedback review, Emily highlig...",Emily flagged delivery delays; Ana's AI propos...,0.40,1.0,0.280000,0.000000
2,"During the HR sync, James mentioned morale iss...",James raised morale issues; Clara's survey pro...,0.00,1.0,0.355556,0.000000
3,The budget alignment meeting revealed overspen...,Pierre and Lisa negotiated R&D budget; milesto...,0.00,1.0,0.428571,0.000000


In [5]:
eval_df.describe()

,polarity_shift,lexical_diversity,rouge,bleu
count,4.000000,4.0,4.000000,4.000000
mean,0.037500,1.0,0.344979,0.006344
std,0.268871,0.0,0.063702,0.012688
min,-0.250000,1.0,0.280000,0.000000
25%,-0.062500,1.0,0.306842,0.000000
50%,0.000000,1.0,0.335673,0.000000
75%,0.100000,1.0,0.373810,0.006344
max,0.400000,1.0,0.428571,0.025377


###  Visualize Polarity Shift and Factual Consistency

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.barplot(data=eval_df[['polarity_shift', 'factual_consistency']])
plt.title('Polarity Shift and Factual Consistency')
plt.ylabel('Score')
plt.show()